# Research Watch updater

Use this **root notebook only**. It is written for the **current combined-site structure** and updates only the root site files. After it finishes, run `python scripts/sync_dual_site.py` to propagate the changes into `/dark/`.

In [ ]:
from pathlib import Path

REPO_ROOT = Path('.').resolve()
WEEKLY_INPUTS_DIR = REPO_ROOT / 'weekly-inputs'
AUTO_PICK_LATEST_DOCX = True
MANUAL_DOCX_PATH = WEEKLY_INPUTS_DIR / '2026-03-your-topic.docx'
REPLACE_DUPLICATE_POST_ID = True

print('Repo root:', REPO_ROOT)
print('weekly-inputs:', WEEKLY_INPUTS_DIR)

In [ ]:
import sys

SCRIPTS_DIR = REPO_ROOT / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from research_watch_updater import (
    parse_weekly_docx,
    normalize_docx_data,
    pick_latest_docx,
    update_site,
)

TEMPLATE_DOCX_HINTS = {
    'weekly-research-watch-template.docx',
    'template.docx',
    'research-watch-template.docx',
}

In [ ]:
docx_path = pick_latest_docx(WEEKLY_INPUTS_DIR, TEMPLATE_DOCX_HINTS) if AUTO_PICK_LATEST_DOCX else MANUAL_DOCX_PATH
raw_data = parse_weekly_docx(docx_path)
notes, normalized_stem = normalize_docx_data(raw_data, docx_path)

print('DOCX selected:', docx_path)
print('Normalized output stem:', normalized_stem)
print('Generated HTML will be:', REPO_ROOT / 'posts' / f'{normalized_stem}.html')
if notes:
    print('')
    print('Normalization notes:')
    for note in notes:
        print('-', note)

print('')
print('Metadata preview:')
for k, v in raw_data['metadata'].items():
    if v:
        print(f'- {k}: {v}')

In [ ]:
result = update_site(
    repo_root=REPO_ROOT,
    docx_path=None if AUTO_PICK_LATEST_DOCX else MANUAL_DOCX_PATH,
    auto_pick_latest=AUTO_PICK_LATEST_DOCX,
    replace_duplicate=REPLACE_DUPLICATE_POST_ID,
)

print('Updated ongoing-work.html:', result.ongoing_path)
print('Updated index.html:', result.index_path)
print('Generated/updated post page:', result.post_path)
print('Inserted/updated Post ID:', result.post_id)
print('')
print('Next step: run -> python scripts/sync_dual_site.py')